In [1]:
from typing import TypedDict, Annotated, Literal
import operator
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
import os

load_dotenv()

True

In [2]:

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [3]:
llm = ChatOpenAI(model=os.getenv('OPENAI_MODEL', 'gpt-4o-mini'))
def chat_node(state: ChatState):
    messages = state['messages']

    response = llm.invoke(messages)

    return {'messages': [response]}


In [4]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [5]:
# Interactive chat loop. The conversation history lives in `state` and is
# passed back into the graph on every turn, so the bot remembers earlier
# messages. Type 'exit', 'quit' or 'bye' to stop.

state: ChatState = {'messages': []}

thread_id = '1'
while True:
    user_input = input('You: ')

    if user_input.strip().lower() in ('exit', 'quit', 'bye'):
        print('Bot: Goodbye!')
        break

    config = {'configurable': {'thread_id': thread_id}}

    # state['messages'].append(HumanMessage(content=user_input))
    # state = chatbot.invoke(state)

    response = chatbot.invoke({'messages': [HumanMessage(content=user_input)]}, config=config)

    print('Bot:', response['messages'][-1].content)

Bot: Hello Subrotho! How can I assist you today?


KeyboardInterrupt: Interrupted by user